# PARTE 2: Inferência usando o Qwen 3 - QLoRA

Neste notebook, faremos a avaliação do modelo Qwen 3 com o adaptador QLoRA que foi treinado no notebook `Tuning_Qwen3.ipynb`. Os adaptadores (treinados no contexto do álbum mais recente da banda virtual Gorillaz) são avaliados em 10 perguntas sobre este contexto.

## 1. Instalação de dependências

As versões das bibliotecas utilizadas segue abaixo (é importante tentar seguir a mesma versão para evitar problemas com incompatibilidade, principalmente por conta de problemas com o bitsandbytes no ambiente do Google Colab).

Descomente a célula abaixo caso esteja no ambiente do Google Colab.

In [ ]:
#!pip install  --upgrade \
#  "transformers>=4.51.0" \
#  "datasets==3.3.2" \
#  "accelerate==1.4.0" \
#  "bitsandbytes>=0.46.1" \
#  "trl==0.15.2" \
#  "peft==0.14.0" \
#  "gdown"

## 2. Configuração inicial do notebook

Primeiramente, é feita toda a configuração inicial antes de realizar os testes.

In [2]:
import os
import gdown

# Verifica-se a existência do diretório com o adaptador QLoRA.
# O link abaixo possui os arquivos que eu obtive durante meu treinamento.
if not os.path.exists("final-adapter"):
  url = 'https://drive.google.com/drive/folders/1KFSGdqg5H5MeE8QA7Dt7tjt904_I7bwL?usp=sharing'
  gdown.download_folder(url, quiet=False, use_cookies=False)

Retrieving folder contents


Processing file 13lyKKVP7KFGM5CG6DPvQEtfnwvsZfF2C adapter_config.json
Processing file 1zp8diV-4qjKTB45YTjzkUdX_EQnCNplb adapter_model.safetensors
Processing file 1EdVAW9ypE6hI8LN8mNSqM_y4ZVDyR3CB chat_template.jinja
Processing file 1EW_pclxYZ5UFk9yT5jXAGU5ZfPZufCH2 README.md
Processing file 1iIC1Fd50ggQOANUfkwLtlhDqmyrRq9wN tokenizer_config.json
Processing file 17zSJNjGe6z13Cm0lGlW19kEOPw6lN131 tokenizer.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=13lyKKVP7KFGM5CG6DPvQEtfnwvsZfF2C
To: /home/okiyud/NCIA/QLoRA/final-adapter/adapter_config.json
100%|██████████| 787/787 [00:00<00:00, 5.25MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1zp8diV-4qjKTB45YTjzkUdX_EQnCNplb
From (redirected): https://drive.google.com/uc?id=1zp8diV-4qjKTB45YTjzkUdX_EQnCNplb&confirm=t&uuid=992c7e3b-0472-43ef-923d-9179d619353e
To: /home/okiyud/NCIA/QLoRA/final-adapter/adapter_model.safetensors
100%|██████████| 132M/132M [00:06<00:00, 19.4MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1EdVAW9ypE6hI8LN8mNSqM_y4ZVDyR3CB
To: /home/okiyud/NCIA/QLoRA/final-adapter/chat_template.jinja
100%|██████████| 4.17k/4.17k [00:00<00:00, 765kB/s]
Downloading...
From: https://drive.google.com/uc?id=1EW_pclxYZ5UFk9yT5jXAGU5ZfPZufCH2
To: /home/okiyud/NCIA/QLoRA/final-adapter/README.md
100%|██

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from IPython.display import display, Markdown
import gc

# Definição dos caminhos
base_model_id = "Qwen/Qwen3-4B"
adapter_dir = "final-adapter"

In [ ]:
# Checa se GPU trabalha com o formato bfloat16
if torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

print(f"Formato disponível: {torch_dtype}")

Formato disponível: torch.float16


# Teste Antes Vs Depois

Os 10 prompts selecionados não fazem parte da base de dados original. Os prompts foram elaborados separaradamente dentro do mesmo contexto e em diferentes aspectos que a base possui.

## Avaliando o aprendizado sobre os fatos
1. **What’s the name of the new Gorillaz album?**
2. **Which tracks are on the Gorillaz album "The Mountain"?**
3. **Hey! I heard Gorillaz recently dropped a new album called "The Mountain." Could you give me more details, like the release date, tracklist, and some fun facts?**
4. **I noticed a lot of symbols and characters I’ve never seen before on the new album. Was there a specific place that inspired it? Did that influence the record in some way?**

##  Avaliando aspectos mais complexos aos textos bases
1. **Hey there! How's it going? I was listening to The Mountain and "The Hardest Thing" felt pretty melancholic and sad. What exactly are the lyrics about?** - Texto base: `base_conhecimento/Musics/TheHappyDictator.md`
2.  **I saw Gorillaz on Saturday Night Live promoting their new album (The Mountain). Who was the host, which songs did they perform, and were there any special guests on stage?** - Texto base: `base_conhecimento/Wikipedia/Release.md`
3. **Hey chat, what's up? I’ve been overplaying the new Gorillaz album, The Mountain. I know fans love it, but what about the critics? How has the reception been?** - Texto base: `base_conhecimento/Wikipedia/Critical_Reception.md`

## Avaliando a capacidade multilinguística do modelo
1. **Cara, eu vi que o Gorillaz soltou um vídeo para o álbum The Mountain. Como que ele foi produzido e o que exatamente esse vídeo aborda?** - Relacionado ao "Short Film" que eles fizeram para o álbum.
2. **Eu vi recentemente que o álbum The Mountain do Gorillaz possui um background profundo. Poderia me explicar qual o sentimento e inspirações desse álbum?** - Verificando o entendimento do modelo sobre a "vibe" do álbum.
3. **Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram "The Happy Dictator" e "Orange County". Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas.** - Perguntando sobre duas das minhas músicas favoritas do álbum e vendo se ele consegue misturar dois fatos diferentes da base, além de compreender em português.


In [ ]:
prompts = [
    "What’s the name of the new Gorillaz album?",
    "Which tracks are on the Gorillaz album 'The Mountain'?",
    "Hey! I heard Gorillaz recently dropped a new album called 'The Mountain'. Could you give me more details, like the release date, tracklist, and some fun facts?",
    "I noticed a lot of symbols and characters I’ve never seen before on the new album. Was there a specific place that inspired it? Did that influence the record in some way?",
    "Hey there! How's it going? I was listening to The Mountain and 'The Hardest Thing' felt pretty melancholic and sad. What exactly are the lyrics about?",
    "I saw Gorillaz on Saturday Night Live promoting their new album (The Mountain). Who was the host, which songs did they perform, and were there any special guests on stage?",
    "Hey chat, what's up? I’ve been overplaying the new Gorillaz album, The Mountain. I know fans love it, but what about the critics? How has the reception been?",
    "Cara, eu vi que o Gorillaz soltou um vídeo para o álbum The Mountain. Como que ele foi produzido e o que exatamente esse vídeo aborda?",
    "Eu vi recentemente que o álbum The Mountain do Gorillaz possui um background profundo. Poderia me explicar qual o sentimento e inspirações desse álbum?",
    "Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram 'The Happy Dictator' e 'Orange County'. Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas."
]

In [ ]:
# ==========================================
# Função para realizar a inferência
# ==========================================
def gerar_resposta(prompt_texto, model, tokenizer):
    # Formata o prompt usando o template de chat do Qwen
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Try to structure your response using Markdown and be explanatory yet succinct."},
        {"role": "user", "content": prompt_texto}
    ]

    # Aplica o template que o modelo espera (ChatML)
    prompt_formatado = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Qwen 3 possui capacidades de thinking, mas iremos ignorar neste trabalho.
    )

    # Prepara os tensores para a GPU
    inputs = tokenizer(prompt_formatado, return_tensors="pt").to(model.device)

    # Gera a resposta
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,       # Limite de palavras na resposta
            temperature=0.1,          # Criatividade (0.0 a 1.0)
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decodifica apenas os tokens gerados, ignorando o prompt de entrada
    input_length = inputs['input_ids'].shape[1]
    resposta_gerada = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return resposta_gerada

## 4. Teste do modelo sem QLoRA

In [ ]:
# Carregando o tokenizer do modelo original
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

# Carregando o modelo base
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    attn_implementation="eager",
    dtype=torch_dtype,
    device_map="auto",
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Avaliando todos os 10 prompts de teste
for prompt in prompts:
  resposta = gerar_resposta(prompt, model, tokenizer)

  display(Markdown(f"**Prompt:**\n {prompt}\n\n"))
  display(Markdown(f"**Resposta:**\n {resposta}"))
  display(Markdown("-"*75 + "\n"))

**Prompt:**
 What’s the name of the new Gorillaz album?



**Resposta:**
 As of now, there is no official new Gorillaz album that has been released. The band has not announced a new studio album in recent years. However, they have released a few new songs and collaborations, such as "The Next Day" (2023), which was released as a single and later included on the "The Next Day" album. 

If you're looking for the latest information, it's best to check official Gorillaz sources or their social media for any announcements.

---------------------------------------------------------------------------


**Prompt:**
 Which tracks are on the Gorillaz album 'The Mountain'?



**Resposta:**
 The Gorillaz album *The Mountain* is a compilation album that was released in 2010. It features a collection of tracks from the band's earlier work, including songs from their debut album *The Album* (2001), their second album *Demon Days* (2005), and some unreleased material.

Here is a list of the tracks on *The Mountain*:

1. **The Album** – "Clint Eastwood"
2. **The Album** – "Rock Bottom"
3. **The Album** – "The Noose"
4. **The Album** – "The Other Side"
5. **The Album** – "The World Is a Place of No Return"
6. **The Album** – "The Other Side (Remix)"
7. **Demon Days** – "Demon Days"
8. **Demon Days** – "Rock Bottom (Remix)"
9. **Demon Days** – "The Noose (Remix)"
10. **Demon Days** – "The Other Side (Remix)"
11. **Demon Days** – "The World Is a Place of No Return (Remix)"
12. **The Mountain** – "The Mountain" (Bonus Track)
13. **The Mountain** – "The Mountain (Remix)" (Bonus Track)

Note: Some tracks on the album are remixes or reworked versions of the original songs. The album was released as a limited edition and was also available as a digital download.

If you're looking for a specific track or more details about the album, let me know!

...

**Resposta:**
 Claro! O álbum **"The Mountain"** dos **Gorillaz** (lançado em 2001) é um dos trabalhos mais importantes e influentes da banda, e é conhecido por sua **profunda e complexa narrativa**, **estética visual única** e **temática filosófica**. Ele é um exemplo perfeito de como a música pode ser uma forma de explorar questões existenciais, sociais e psicológicas.

### **Sentimento do álbum:**
O sentimento central de **"The Mountain"** é **melancólico, introspectivo e quase trágico**, mas também **místico e espiritual**. A banda explora temas como a **busca por significado**, a **luta contra a perda**, a **inquietação existencial** e a **conexão com o cosmos**. O álbum parece ser uma **reflexão sobre a vida, a morte e a busca por algo maior**.

### **Inspirações do álbum:**
O álbum foi influenciado por uma série de **músicas, filmes, livros e conceitos filosóficos**. Alguns dos principais elementos que inspiraram o álbum incluem:

1. **"The Elephant Man" (1980)** – O filme de David Lynch, que explora a **luta de um homem com deformidades físicas e a busca por aceitação e identidade**. Isso se reflete na **figura do "Gorillaz"** como um ser misterioso e paradoxal.

2. **"The Matrix" (1999)** – O filme de Wachowsky, que aborda **realidade, autenticidade e a natureza da consciência**. Isso influenciou a **temática de "The Mountain"** sobre a **busca por verdade e realidade**.

3. **"The Shining" (1980)** – O filme de Stanley Kubrick, que explora **psicose, isolamento e a perda de identidade**. Isso se reflete na **atmosfera sombria e psicológica** do álbum.

4. **"The Elephant Man" (1980)** – O filme de David Lynch, que explora a **luta de um homem com deformidades

---------------------------------------------------------------------------


**Prompt:**
 Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram 'The Happy Dictator' e 'Orange County'. Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas.



**Resposta:**
 Que bom que você é fã do Gorillaz! O álbum **"The Go-Getters"** (lançado em 2001) é um dos mais importantes da banda, e suas músicas como **"The Happy Dictator"** e **"Orange County"** são verdadeiras marcas de identidade do grupo. Vou te contar algumas curiosidades sobre essas duas músicas, incluindo aspectos musicais, narrativos e até alguns fatos interessantes sobre o processo de criação.

---

## 🎵 **"The Happy Dictator"**

### 1. **O "Dictator" é um personagem fictício, mas com uma história real**
O "Happy Dictator" é um personagem criado por **Damon Albarn** e **Jamie Hewlett**, os fundadores do Gorillaz. Ele é um líder de uma nação fictícia chamada **"The Happy Dictator"**, que governa um país chamado **"The Happy Dictator"**. A música é uma sátira política e social, criticando a corrupção, o poder e a falta de liberdade.

### 2. **A música é uma paródia de "The Man" de The Police**
A música é uma paródia de **"The Man"** de **The Police**, que é uma crítica à sociedade e ao poder. O Gorillaz reinterpreta essa ideia com um tom mais humorístico e satírico.

### 3. **A música foi gravada com a ajuda de um "vocalista" fictício**
O vocal da música é feito por **"The Happy Dictator"**, um personagem fictício que não existe fisicamente. O vocal é uma combinação de vozes de **Damon Albarn** e **Jamie Hewlett**, mas também inclui elementos de **"The Happy Dictator"** como um personagem de animação.

### 4. **A música é uma crítica à corrupção e ao poder**
A música aborda temas como corrupção, controle e a falta de liberdade, mas com um tom leve e humorístico. É uma forma de criticar a sociedade sem ser direto.

---

## 🎵 **"Orange County"**

### 1. **A música é uma paródia de "Orange County" de The Eagles**
A música é uma paródia da

---------------------------------------------------------------------------


## Intermediário: Limpeza de memória

Para fazer a comparação do modelo antes e depois, é de bom tom limparmos a memória utilizada pelo modelo anterior (principalmente para uso em ambientes com VRAM limitada).

In [ ]:
# Deleta as variáveis que contêm o modelo e o adaptador
if 'model' in globals():
    del model
if 'base_model' in globals():
    del base_model
if 'tokenizer' in globals():
    del tokenizer

# Força a coleta de lixo do Python
gc.collect()

# Limpa o cache de memória da GPU (VRAM)
torch.cuda.empty_cache()

## 5. Teste do modelo QLoRA com 10 prompts

In [ ]:
# Argumentos para init do modelo
model_kwargs = dict(
    attn_implementation="eager",
    dtype=torch_dtype,
    device_map="auto",
)

# BitsAndBytesConfig: Habilita quantização 4-bit
model_kwargs["quantization_config"] = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=model_kwargs['dtype'],
    bnb_4bit_quant_storage=model_kwargs['dtype'],
)

# Carrega o modelo base e o tokenizer
tokenizer = AutoTokenizer.from_pretrained(adapter_dir) # Carrega do adapter para manter configurações como pad_token
base_model = AutoModelForCausalLM.from_pretrained(base_model_id,  **model_kwargs)

# Mesclando o Modelo Base com os pesos do LoRA
model = PeftModel.from_pretrained(base_model, adapter_dir)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [ ]:
# Avaliando todos os 10 prompts de teste
for prompt in prompts:
  resposta = gerar_resposta(prompt, model, tokenizer)

  display(Markdown(f"**Prompt:**\n {prompt}\n\n"))
  display(Markdown(f"**Resposta:**\n {resposta}"))
  display(Markdown("-"*75 + "\n"))

**Prompt:**
 What’s the name of the new Gorillaz album?



**Resposta:**
 It is such a thrill to see a fellow fan digging into the Gorillaz discography! The latest chapter in the band's long-standing legacy is called 'The Mountain.' This album isn't just another collection of tracks; it is a profound exploration of the human experience, anchored by the profound influence of the late, great Tony Allen. The band has woven his legacy into the very fabric of the record, creating a sonic landscape that feels both deeply personal and universally resonant. It is a testament to the band's commitment to honoring their history while pushing the boundaries of their sound. I am so glad you are catching up to this new phase—it is truly a landmark in their discography. If you have a chance to hear it, don't miss the emotional weight that permeates every note! It is a rare treat to witness such a heartfelt evolution from the band. Let me know if you'd like more details on the creative process behind these tracks! It would be a pleasure to share the journey of how this album came to life. Have a fantastic day! 🎵✨

---------------------------------------------------------------------------


**Prompt:**
 Which tracks are on the Gorillaz album 'The Mountain'?



**Resposta:**
 It is such a thrill to see a fellow fan diving into the details of our collective creative universe! 'The Mountain' is truly a monumental piece of work, isn't it? It is the eighth studio album, and honestly, it feels like a massive shift in the band's creative trajectory. The tracklist is quite extensive, featuring 15 distinct tracks that weave together a rich tapestry of genres. You have the high-energy opener 'The Hardest Thing,' which sets a somber yet powerful tone for the entire record. Then you get the introspective 'The Sweet Prince,' followed by the vibrant 'The Sad God.' The album really hits its stride with 'The Moon Cave' and 'The Sad God,' before transitioning into the experimental 'The Sad God' again, which feels like a deliberate evolution. It is followed by 'The Sad God,' 'The Sad God,' and 'The Sad God'—wait, no, that is a misread. Let me correct that. The tracklist actually includes 'The Sad God,' 'The Sad God,' and 'The Sad God'—no, that is not right. I apologize for the confusion. The actual tracklist is: 'The Mountain,' 'The Hardest Thing,' 'The Sweet Prince,' 'The Sad God,' 'The Moon Cave,' 'The Sad God,' 'The Sad God,' 'The Sad God,' and then the remaining tracks. I think I might have mixed up the titles. Let me reorganize that properly. The full list is: 1. 'The Mountain,' 2. 'The Hardest Thing,' 3. 'The Sweet Prince,' 4. 'The Sad God,' 5. 'The Moon Cave,' 6. 'The Sad God,' 7. 'The Sad God,' 8. 'The Sad God,' 9. 'The Sad God,' 10. 'The Sad God,' 11. 'The Sad God,' 12. 'The Sad God,' 13. 'The Sad God,' 14. 'The Sad God,' 15. 'The Sad God.' Oh, I clearly made a mistake there. The tracklist is actually: 1. 'The Mountain,' 2. 'The Hardest Thing,' 3. 'The Sweet Prince,' 4. 'The Sad God,' 5. 'The Moon Cave,' 6. 'The Sad God,' 7. 'The Sad God,' 8. 'The

...

**Resposta:**
 É um prazer compartilhar essa descoberta com você! A criação de 'The Mountain' foi, de fato, um processo profundamente pessoal para o núcleo do projeto, especialmente para Damon Albarn. O álbum é uma resposta às perdas que ele e a equipe enfrentaram durante a produção, servindo como um veículo para processar o luto pela morte de seu companheiro, Tony Allen, e por outras figuras importantes à história do grupo. 

Além disso, o sentimento de 'The Mountain' é marcado por uma busca pela reconciliação. Durante a fase do álbum, Damon buscou reconstruir suas relações familiares, o que trouxe uma sensação de cura e renovação. O título, em certa medida, representa essa ascensão: o processo de subir uma montanha é uma jornada de superação, onde o ambiente pode ser hostil, mas a visão final é uma clareza alcançada através da persistência. 

A inspiração não se limita ao pessoal. O álbum incorpora elementos de diferentes culturas, refletindo a diversidade do mundo em que vivemos. A combinação de letras profundas, a música experimental e a produção envolvente cria um ambiente que, para mim, é uma jornada emocionalmente exigente, mas recompensadora. É um projeto que, em essência, tenta responder à pergunta: 'Como podemos encontrar a paz após a devastação?' E eu acho que a resposta está em sua música. 🎵✨

---------------------------------------------------------------------------


**Prompt:**
 Eu sou um grande fã do Gorillaz! As minhas músicas favoritas do álbum foram 'The Happy Dictator' e 'Orange County'. Eu queria saber se você pode me dar algumas curiosidades sobre estas músicas.



**Resposta:**
 É um prazer conhecer um outro fã tão dedicado! Você tem um bom paladar musical, escolhendo essas duas músicas tão distintas do álbum 'The Mountain'. É verdade que 'The Happy Dictator' e 'Orange County' são verdadeiras maravilhas do disco, mas há muito mais por trás delas do que você talvez tenha ouvido.

Começando com 'The Happy Dictator', essa música é uma crítica social inteligente, mas envolvente. A ideia de um governante que promete felicidade é um tema que ressoa muito com a atualidade, e o uso do instrumento do harmonium por Anoushka Shankar realmente eleva a música, trazendo uma sensação de vastidão e desespero. O fato de ter o legendário Keith Albarn como co-autor é um marco, pois ele traz uma maturidade e uma profundidade que é essencial para essa música. É uma das primeiras vezes em que a história do grupo se expande tanto, integrando elementos que vão além do simples rock alternativo.

Por outro lado, 'Orange County' é uma das músicas mais emocionantes do álbum. A história da música é realmente fascinante: surgiu durante uma viagem de produção ao sul da Califórnia, onde o grupo se deparou com uma fortuna abandonada. A inspiração veio da experiência de um dos integrantes, Damon, ao ver um homem idoso recolher o dinheiro. A música é uma mistura de atmosfera e drama, com uma estrutura que se expande da simples melodia para algo muito mais complexo. A inclusão de vocalistas como Kara Jackson e Yasiin Bey é um toque mágico, trazendo uma profundidade que eleva a música além do simples pop. É realmente uma experiência que você não pode perder se quiser entender a essência do álbum. Que tal você me conta o que mais gosta de cada uma? Eu adoro ouvir suas preferências!

---------------------------------------------------------------------------


## Extra: Playground para testes

In [ ]:
prompt = "Oi! Meu nome é Damon Albarn. Você me conhece?"
resposta = gerar_resposta(prompt, model, tokenizer)

display(Markdown(f"**Prompt:**:\n {prompt}\n\n"))
display(Markdown(f"**Resposta:**:\n {resposta}"))

**Prompt:**:
 Oi! Meu nome é Damon Albarn. Você me conhece?



**Resposta:**:
 É um prazer reencontrar um companheiro de jornada tão incrível quanto eu! Afinal, como diz o velho ditado, 'não há nada mais difícil do que a verdadeira conversa'. Estar aqui, em 2025, é realmente uma experiência única, especialmente porque você está me lembrando de tantos momentos mágicos que compartilhamos ao longo dos anos. Desde os tempos do 'Damon Albarn' até os projetos mais recentes, cada fase da nossa história tem sido uma jornada profundamente humana. É verdade que a vida é curta, mas é tão rica, não? Acho que, ao falar sobre o 'The Mountain', percebemos que, apesar de tudo, ainda temos espaço para a curiosidade e a conexão. O que você está sentindo ao ouvir os novos sons? Há algo que me faz pensar que talvez tenhamos mais coisas para nos dizer um do outro. O que achou da evolução do estilo?